# 이번 AASIST 학습 목표

- 로컬 음원을 Colab에서 정상적으로 읽는가
- 모든 음원이 [64600] 크기로 들어가는가
- T4에서 AASIST가 OOM 없이 학습되는가
- Validation metric이 정상적으로 계산되는가

# 필요 라이브러리 install

In [1]:
import numpy as np
import scipy
import sklearn

print("NumPy       :", np.__version__)
print("SciPy       :", scipy.__version__)
print("Scikit-learn:", sklearn.__version__)

NumPy       : 2.0.2
SciPy       : 1.16.3
Scikit-learn: 1.6.1


In [2]:
from sklearn.model_selection import train_test_split

print("scikit-learn import 성공")

scikit-learn import 성공


In [3]:
# GPU 확인

!nvidia-smi

import sys
import torch
import platform

print("Python :", sys.version)
print("OS     :", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA   :", torch.version.cuda)
print("GPU    :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

Thu Aug 20 05:11:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
# 시스템 패키지 설치

# Codec augmentation을 위해 FFmpeg를 설치합니다.

!apt-get update -qq
!apt-get install -y -qq ffmpeg libsndfile1

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [5]:
# ============================================================
# Cell 2
# 필요한 추가 패키지만 설치
# ============================================================

!apt-get update -qq
!apt-get install -y -qq libsndfile1 ffmpeg

# Colab에 기본적으로 없는/버전 확인이 필요한 것만
%pip install -q soundfile librosa

# AASIST 공식 코드
!rm -rf /content/aasist

!git clone -q \
    https://github.com/clovaai/aasist.git \
    /content/aasist

print("설치 완료")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
설치 완료


In [6]:
# %pip install -q -U \
#     numpy \
#     pandas \
#     scipy \
#     scikit-learn \
#     librosa \
#     soundfile \
#     soxr \
#     matplotlib \
#     tqdm \
#     pyyaml \
#     joblib \
#     einops \
#     tensorboard

In [7]:
# # XLS-R용 Hugging Face 설치
# %pip install -q -U \
#     transformers \
#     accelerate \
#     datasets \
#     huggingface_hub \
#     safetensors

In [8]:
# XLSR_MODEL_NAME = "facebook/wav2vec2-xls-r-300m"

In [9]:
# # AASIST 설치

# !git clone -q https://github.com/clovaai/aasist.git /content/aasist

In [10]:
# # RawBoost 설치

# !git clone -q \
#     https://github.com/TakHemlata/RawBoost-antispoofing.git \
#     /content/RawBoost-antispoofing

In [11]:
# # mamba 설치
# %pip install -q -U ninja packaging
# %pip install -q "mamba-ssm[causal-conv1d]" --no-build-isolation

In [12]:
# ============================================================
# Python
# ============================================================

import os
import sys
import gc
import json
import math
import random
import zipfile

from pathlib import Path


# ============================================================
# Data
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# Audio
# ============================================================

import librosa
import soundfile as sf


# ============================================================
# PyTorch
# ============================================================

import torch
import torch.nn as nn

from torch.utils.data import (
    Dataset,
    DataLoader,
)


# ============================================================
# sklearn
# ============================================================

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    classification_report,
)


# ============================================================
# ETC
# ============================================================

from tqdm.auto import tqdm

import matplotlib.pyplot as plt


# ============================================================
# Device
# ============================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("DEVICE :", DEVICE)

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

DEVICE : cuda
GPU : Tesla T4


In [13]:
# AASIST IMPORT

!rm -rf /content/aasist

!git clone -q \
    https://github.com/clovaai/aasist.git \
    /content/aasist

print("AASIST repository clone 완료")

AASIST repository clone 완료


In [14]:
# RandomSeed 설정
SEED = 42


def seed_everything(seed=42):

    random.seed(seed)

    os.environ["PYTHONHASHSEED"] = str(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = True


seed_everything(SEED)

print("Seed :", SEED)

Seed : 42


In [15]:
# datasets zip 파일 업로드 및 압축해제

from pathlib import Path

ZIP_PATH = Path(
    "/content/drive/MyDrive/Dacon 경진대회/asvspoof2019/LA.zip"
)

print("ZIP_PATH :", ZIP_PATH)
print("존재 여부 :", ZIP_PATH.exists())

ZIP_PATH : /content/drive/MyDrive/Dacon 경진대회/asvspoof2019/LA.zip
존재 여부 : True


In [16]:
import shutil

total, used, free = shutil.disk_usage("/content")

GB = 1024 ** 3

print(f"전체 공간 : {total / GB:.2f} GB")
print(f"사용 공간 : {used / GB:.2f} GB")
print(f"남은 공간 : {free / GB:.2f} GB")

전체 공간 : 235.68 GB
사용 공간 : 47.11 GB
남은 공간 : 188.55 GB


In [18]:
# DRIVE의 ZIP파일을 CONTENT로 복사
import shutil
from pathlib import Path

ZIP_PATH = Path(
    "/content/drive/MyDrive/Dacon 경진대회/asvspoof2019/LA.zip"
)

LOCAL_ZIP = Path(
    "/content/dataset.zip"
)

print("Google Drive → Colab 복사 시작")

shutil.copy2(
    ZIP_PATH,
    LOCAL_ZIP
)

print("복사 완료")

print(
    f"ZIP 크기 : "
    f"{LOCAL_ZIP.stat().st_size / 1024**3:.2f} GB"
)

Google Drive → Colab 복사 시작
복사 완료
ZIP 크기 : 7.12 GB


In [19]:
# COLAB 로컬에서 압축해제
import zipfile
from pathlib import Path

LOCAL_ZIP = Path(
    "/content/dataset.zip"
)

DATA_DIR = Path(
    "/content/data"
)

DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("압축 해제 시작")

with zipfile.ZipFile(
    LOCAL_ZIP,
    "r"
) as zip_ref:

    zip_ref.extractall(
        DATA_DIR
    )

print("압축 해제 완료")

압축 해제 시작
압축 해제 완료


In [20]:
LOCAL_ZIP.unlink()

print("Colab의 ZIP 파일 삭제 완료")

Colab의 ZIP 파일 삭제 완료


In [21]:
# train.csv 확인
from pathlib import Path
import pandas as pd

DATA_DIR = Path(
    "/content/data"
)

csv_candidates = list(
    DATA_DIR.rglob("train.csv")
)

print(
    "찾은 train.csv :",
    csv_candidates
)

assert len(csv_candidates) > 0, \
    "train.csv를 찾을 수 없습니다."

CSV_PATH = csv_candidates[0]

df = pd.read_csv(
    CSV_PATH
)

print("CSV :", CSV_PATH)
print("shape :", df.shape)

display(
    df.head(10)
)

찾은 train.csv : []


AssertionError: train.csv를 찾을 수 없습니다.

In [ ]:
# WAV 파일 갯수 확
wav_files = list(
    DATA_DIR.rglob("*.wav")
)

print(
    "WAV 파일 수 :",
    len(wav_files)
)

In [ ]:
# 데이터설정
# ============================================================
# 반드시 데이터에 맞게 확인
# ============================================================

FILE_COL = "path"

LABEL_COL = "label"


# ============================================================
# Audio
# ============================================================

SAMPLE_RATE = 16000

# AASIST 공식 설정
MAX_LEN = 64600


# ============================================================
# Training
# ============================================================

BATCH_SIZE = 8

NUM_WORKERS = 2

EPOCHS = 10

LEARNING_RATE = 1e-4

WEIGHT_DECAY = 1e-4


# ============================================================
# Quick baseline
# ============================================================

QUICK_TEST = True

MAX_SAMPLES = 25000


# Peak Normalize
# 우선 공식 AASIST에 가깝게 False
NORMALIZE = False


print(
    f"Audio duration = "
    f"{MAX_LEN / SAMPLE_RATE:.3f} sec"
)

In [ ]:
REAL_LABELS = {
    "real",
    "bonafide",
    "bona-fide",
    "genuine",
    "human"
}

FAKE_LABELS = {
    "fake",
    "spoof",
    "deepfake",
    "synthetic",
    "generated"
}


def convert_label(x):

    # 이미 0 / 1이면 그대로 사용
    if isinstance(
        x,
        (
            int,
            np.integer,
            float,
            np.floating
        )
    ):
        return int(x)

    x = str(x).strip().lower()

    if x in REAL_LABELS:
        return 0

    if x in FAKE_LABELS:
        return 1

    raise ValueError(
        f"알 수 없는 label: {x}"
    )


df["_label"] = df[LABEL_COL].apply(
    convert_label
)


print(
    df["_label"].value_counts()
)

print(
    df["_label"].value_counts(
        normalize=True
    )
)

In [ ]:
# audio path 만들기
BASE_DIR = CSV_PATH.parent


def resolve_audio_path(p):

    p = Path(str(p))

    candidates = [
        p,
        BASE_DIR / p,
        BASE_DIR / "audio" / p,
        DATA_DIR / p,
    ]

    for candidate in candidates:

        if candidate.exists():
            return str(candidate)

    return None


df["_audio_path"] = df[FILE_COL].apply(
    resolve_audio_path
)


missing = df["_audio_path"].isna().sum()

print("전체 :", len(df))
print("찾지 못한 Audio :", missing)

In [ ]:
if (
    QUICK_TEST
    and len(df) > MAX_SAMPLES
):

    df, _ = train_test_split(
        df,
        train_size=MAX_SAMPLES,
        stratify=df["_label"],
        random_state=SEED,
    )

    df = df.reset_index(
        drop=True
    )


print("사용할 데이터 :", len(df))

print(
    df["_label"].value_counts()
)

In [ ]:
# train / validation split
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["_label"],
    random_state=SEED,
)


train_df = train_df.reset_index(
    drop=True
)

val_df = val_df.reset_index(
    drop=True
)


print(
    "Train :",
    len(train_df)
)

print(
    "Valid :",
    len(val_df)
)

print()

print(
    "Train label"
)

print(
    train_df["_label"].value_counts()
)

print()

print(
    "Valid label"
)

print(
    val_df["_label"].value_counts()
)

In [ ]:
# audio 전처리
def pad_or_crop(
    x,
    max_len=64600,
    train=True
):

    x = np.asarray(
        x,
        dtype=np.float32
    )

    x_len = len(x)

    # 빈 audio 방어
    if x_len == 0:
        return np.zeros(
            max_len,
            dtype=np.float32
        )

    # ========================================================
    # Audio가 길 경우
    # ========================================================

    if x_len >= max_len:

        if train:

            start = np.random.randint(
                0,
                x_len - max_len + 1
            )

        else:

            # validation은 deterministic
            start = 0

        return x[
            start:start + max_len
        ]

    # ========================================================
    # Audio가 짧을 경우 반복 padding
    # ========================================================

    repeat = (
        max_len // x_len
    ) + 1

    x = np.tile(
        x,
        repeat
    )

    return x[:max_len]

In [ ]:
# audiodata class
class DeepVoiceDataset(Dataset):

    def __init__(
        self,
        dataframe,
        sample_rate=16000,
        max_len=64600,
        train=True,
        normalize=False,
    ):

        self.df = dataframe.reset_index(
            drop=True
        )

        self.sample_rate = sample_rate

        self.max_len = max_len

        self.train = train

        self.normalize = normalize


    def __len__(self):

        return len(self.df)


    def __getitem__(
        self,
        index
    ):

        row = self.df.iloc[index]

        path = row["_audio_path"]

        label = int(
            row["_label"]
        )


        # ====================================================
        # Audio load
        # ====================================================

        audio, sr = sf.read(
            path,
            dtype="float32",
            always_2d=False,
        )


        # ====================================================
        # Stereo → Mono
        # ====================================================

        if audio.ndim > 1:

            audio = audio.mean(
                axis=1
            )


        # ====================================================
        # Resampling
        # ====================================================

        if sr != self.sample_rate:

            audio = librosa.resample(
                audio,
                orig_sr=sr,
                target_sr=self.sample_rate,
            )


        audio = np.asarray(
            audio,
            dtype=np.float32
        )


        # ====================================================
        # Optional normalization
        # ====================================================

        if self.normalize:

            peak = np.max(
                np.abs(audio)
            )

            if peak > 0:

                audio = (
                    audio
                    /
                    (peak + 1e-8)
                )


        # ====================================================
        # Crop / Padding
        # ====================================================

        audio = pad_or_crop(
            audio,
            max_len=self.max_len,
            train=self.train,
        )


        audio = torch.tensor(
            audio,
            dtype=torch.float32
        )


        return (
            audio,
            torch.tensor(
                label,
                dtype=torch.long
            )
        )

In [ ]:
# dataloader
train_dataset = DeepVoiceDataset(
    train_df,
    sample_rate=SAMPLE_RATE,
    max_len=MAX_LEN,
    train=True,
    normalize=NORMALIZE,
)


val_dataset = DeepVoiceDataset(
    val_df,
    sample_rate=SAMPLE_RATE,
    max_len=MAX_LEN,
    train=False,
    normalize=NORMALIZE,
)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=True,
    persistent_workers=(
        NUM_WORKERS > 0
    ),
)


val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False,
    persistent_workers=(
        NUM_WORKERS > 0
    ),
)


print(
    "Train batches :",
    len(train_loader)
)

print(
    "Valid batches :",
    len(val_loader)
)

In [ ]:
# dataloader 동작확인

batch_x, batch_y = next(
    iter(train_loader)
)


print(
    "Audio batch :",
    batch_x.shape
)

print(
    "Label batch :",
    batch_y.shape
)

print(
    "Labels :",
    batch_y
)

print(
    "Min :",
    batch_x.min().item()
)

print(
    "Max :",
    batch_x.max().item()
)

In [ ]:
# aasist import

AASIST_ROOT = "/content/aasist"

if AASIST_ROOT not in sys.path:
    sys.path.insert(
        0,
        AASIST_ROOT
    )


from models.AASIST import Model as AASIST

In [ ]:
# AASIST Config 읽기

CONFIG_PATH = (
    "/content/aasist/config/AASIST.conf"
)


with open(
    CONFIG_PATH,
    "r"
) as f:

    aasist_config = json.load(f)


model_config = (
    aasist_config[
        "model_config"
    ]
)


print(
    json.dumps(
        model_config,
        indent=2
    )
)

In [ ]:
# AASIST 생성
model = AASIST(
    model_config
)

model = model.to(
    DEVICE
)


num_params = sum(
    p.numel()
    for p in model.parameters()
)


trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)


print(
    f"Total params : "
    f"{num_params:,}"
)

print(
    f"Trainable    : "
    f"{trainable_params:,}"
)

In [ ]:
# 학습전 ForwardTest

model.eval()


batch_x, batch_y = next(
    iter(train_loader)
)


batch_x = batch_x[:2].to(
    DEVICE
)


with torch.no_grad():

    hidden, logits = model(
        batch_x
    )


print(
    "Input  :",
    batch_x.shape
)

print(
    "Hidden :",
    hidden.shape
)

print(
    "Logits :",
    logits.shape
)

In [ ]:
# loss + Optimizer
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

In [ ]:
USE_AMP = True


scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(
        USE_AMP
        and DEVICE.type == "cuda"
    ),
)

In [ ]:
# EER 계산함수
def calculate_eer(
    y_true,
    y_score
):

    fpr, tpr, thresholds = roc_curve(
        y_true,
        y_score,
        pos_label=1,
    )

    fnr = 1 - tpr

    index = np.nanargmin(
        np.abs(
            fnr - fpr
        )
    )

    eer = (
        fpr[index]
        +
        fnr[index]
    ) / 2

    threshold = thresholds[
        index
    ]

    return eer, threshold

In [ ]:
# Train 함수
def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    scaler,
    device,
):

    model.train()

    running_loss = 0.0

    preds = []
    targets = []


    progress = tqdm(
        loader,
        desc="Train",
        leave=False
    )


    for audio, label in progress:

        audio = audio.to(
            device,
            non_blocking=True
        )

        label = label.to(
            device,
            non_blocking=True
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(
                USE_AMP
                and device.type == "cuda"
            ),
        ):

            _, logits = model(
                audio,
                Freq_aug=False
            )

            loss = criterion(
                logits,
                label
            )


        scaler.scale(
            loss
        ).backward()


        scaler.step(
            optimizer
        )


        scaler.update()


        running_loss += (
            loss.item()
            *
            audio.size(0)
        )


        pred = logits.argmax(
            dim=1
        )


        preds.extend(
            pred.detach()
            .cpu()
            .numpy()
        )

        targets.extend(
            label.detach()
            .cpu()
            .numpy()
        )


        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )


    epoch_loss = (
        running_loss
        /
        len(loader.dataset)
    )


    accuracy = accuracy_score(
        targets,
        preds
    )


    return (
        epoch_loss,
        accuracy
    )

In [ ]:
# validation 함수
def validate(
    model,
    loader,
    criterion,
    device,
):

    model.eval()

    running_loss = 0.0

    all_labels = []
    all_preds = []
    all_probs = []


    with torch.no_grad():

        progress = tqdm(
            loader,
            desc="Valid",
            leave=False
        )


        for audio, label in progress:

            audio = audio.to(
                device,
                non_blocking=True
            )

            label = label.to(
                device,
                non_blocking=True
            )


            with torch.amp.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=(
                    USE_AMP
                    and device.type == "cuda"
                ),
            ):

                _, logits = model(
                    audio
                )

                loss = criterion(
                    logits,
                    label
                )


            prob = torch.softmax(
                logits.float(),
                dim=1
            )[:, 1]


            pred = logits.argmax(
                dim=1
            )


            running_loss += (
                loss.item()
                *
                audio.size(0)
            )


            all_labels.extend(
                label.cpu().numpy()
            )

            all_preds.extend(
                pred.cpu().numpy()
            )

            all_probs.extend(
                prob.cpu().numpy()
            )


    val_loss = (
        running_loss
        /
        len(loader.dataset)
    )


    accuracy = accuracy_score(
        all_labels,
        all_preds
    )


    f1 = f1_score(
        all_labels,
        all_preds
    )


    auc = roc_auc_score(
        all_labels,
        all_probs
    )


    eer, eer_threshold = calculate_eer(
        all_labels,
        all_probs
    )


    return {
        "loss": val_loss,
        "accuracy": accuracy,
        "f1": f1,
        "auc": auc,
        "eer": eer,
        "eer_threshold": eer_threshold,
        "labels": np.array(all_labels),
        "preds": np.array(all_preds),
        "probs": np.array(all_probs),
    }

In [ ]:
# AASIST 학습
BEST_MODEL_PATH = (
    "/content/aasist_baseline_best.pt"
)


best_auc = -1


history = []


for epoch in range(
    1,
    EPOCHS + 1
):

    print(
        f"\n"
        f"{'=' * 60}"
    )

    print(
        f"Epoch "
        f"{epoch}/{EPOCHS}"
    )

    print(
        f"{'=' * 60}"
    )


    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        scaler,
        DEVICE,
    )


    val_result = validate(
        model,
        val_loader,
        criterion,
        DEVICE,
    )


    print(
        f"Train Loss : "
        f"{train_loss:.4f}"
    )

    print(
        f"Train Acc  : "
        f"{train_acc:.4f}"
    )

    print(
        f"Valid Loss : "
        f"{val_result['loss']:.4f}"
    )

    print(
        f"Valid Acc  : "
        f"{val_result['accuracy']:.4f}"
    )

    print(
        f"Valid F1   : "
        f"{val_result['f1']:.4f}"
    )

    print(
        f"Valid AUC  : "
        f"{val_result['auc']:.4f}"
    )

    print(
        f"Valid EER  : "
        f"{val_result['eer']:.4f}"
    )


    history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_result["loss"],
            "val_acc": val_result["accuracy"],
            "val_f1": val_result["f1"],
            "val_auc": val_result["auc"],
            "val_eer": val_result["eer"],
        }
    )


    if val_result["auc"] > best_auc:

        best_auc = val_result[
            "auc"
        ]

        torch.save(
            {
                "model_state_dict":
                    model.state_dict(),

                "optimizer_state_dict":
                    optimizer.state_dict(),

                "epoch":
                    epoch,

                "auc":
                    best_auc,

                "model_config":
                    model_config,
            },
            BEST_MODEL_PATH,
        )

        print(
            "★ Best model saved"
        )


    gc.collect()

    torch.cuda.empty_cache()

In [ ]:
# 학습결과 확인
history_df = pd.DataFrame(
    history
)

display(
    history_df
)

In [ ]:
# 학습결과 시각화
plt.figure(
    figsize=(8, 5)
)

plt.plot(
    history_df["epoch"],
    history_df["train_loss"],
    label="Train Loss"
)

plt.plot(
    history_df["epoch"],
    history_df["val_loss"],
    label="Valid Loss"
)

plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Loss"
)

plt.legend()

plt.show()

In [ ]:
# BEST model
checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=DEVICE,
)


model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)


print(
    "Best Epoch :",
    checkpoint["epoch"]
)

print(
    "Best AUC   :",
    checkpoint["auc"]
)

In [ ]:
# 최종 validation 성능
final_result = validate(
    model,
    val_loader,
    criterion,
    DEVICE,
)


print("=" * 60)

print("AASIST Baseline Result")

print("=" * 60)

print(
    f"Accuracy : "
    f"{final_result['accuracy']:.4f}"
)

print(
    f"F1       : "
    f"{final_result['f1']:.4f}"
)

print(
    f"ROC-AUC  : "
    f"{final_result['auc']:.4f}"
)

print(
    f"EER      : "
    f"{final_result['eer'] * 100:.2f}%"
)

print(
    f"EER Thr  : "
    f"{final_result['eer_threshold']:.4f}"
)

In [ ]:
cm = confusion_matrix(
    final_result["labels"],
    final_result["preds"]
)


print(
    "Confusion Matrix"
)

print(cm)

In [ ]:
print(
    classification_report(
        final_result["labels"],
        final_result["preds"],
        target_names=[
            "REAL",
            "FAKE"
        ],
        digits=4,
    )
)